# 2 · Finding what matches

**The problem.** The same rules that check data also *select* it: "the adults", "each contact with each of their
numbered phones". Writing those selections again as loops repeats the rules, and repeats their mistakes. A predicate
is already a query: its matches are what it selects.

This case study runs predicates as queries over a store, and looks at how the query is planned.

In [1]:
from mbse.Expressions import Expressions as E
from mbse.Expressions.Dialects.Python import Text
from mbse.Patterns import Predicates, Queries
from toolkit import Contact, Phone, book, listed

store = book()
ann = (store.Contact().name("Ann").age(34)
       .phones(lambda e: e.label("home").phone(lambda p: p.number("555-0100")))
       .phones(lambda e: e.label("work").phone(lambda p: p.number("555-0199"))).create())
bob = store.Contact().name("Bob").phones(lambda e: e.label("work").phone(lambda p: p)).create()
kid = store.Contact().name("Kid").age(9).create()
listed(store, ann, bob, kid)
stray = store.Contact().name("Stray").age(50).create()  # built, but never listed
query = Queries.Scan(store)

## Step 1: a predicate's matches

`Queries.Scan(store)` makes any store queryable. `select` streams the matches for which the rule holds, each a mapping
of symbols to objects:

In [2]:
IsAnAdult = (Predicates.OfPredicate.Builder().name("IsAnAdult").symbols({"the": Contact})
             .requires(Text.FromFunction(lambda the: the.age >= 18)).create())
[m["the"].name for m in query.select(IsAnAdult)]

['Ann']

Only Ann. Bob's age is unknown, so he isn't selected unless you ask (`unknown=True`). The stray contact isn't selected
either: a query sees the store's **data**, what its singleton directory reaches, and the stray was never listed.

In [3]:
[m["the"].name for m in query.select(IsAnAdult, unknown=True)]

['Ann', 'Bob']

`select` returns an iterator, and reads the store only as you ask for matches: the first `next` does the work.

## Step 2: variables

A rule can name values it doesn't bind, **variables**, which the query gives:

In [4]:
Older = (Predicates.OfPredicate.Builder().name("Older").symbols({"the": Contact})
         .requires(E.variable("the").age.ge(E.variable("least"))).create())
[m["the"].name for m in query.select(Older, {"least": 5})]

['Ann', 'Kid']

## Step 3: two symbols, and the plan

With two symbols, the matches are every combination of a contact and a phone, filtered by the rule. Scanning every
combination would be slow; the rule's shape says what's practical. `explain` shows the plan:

In [5]:
c, p = E.variable("c"), E.variable("p")
Numbered = (Predicates.OfPredicate.Builder().name("Numbered").symbols({"c": Contact, "p": Phone})
            .requires(Predicates.Contains(c.phones, lambda e: e.phone == p)).requires(p.has("number")).create())
query.explain(Numbered)

['c: scan Contact', 'p: c.phones to phone, any number, then 2 tests']

In [6]:
[(m["c"].name, m["p"].number) for m in query.select(Numbered)]

[('Ann', '555-0100'), ('Ann', '555-0199')]

The `Contains` conjunct relates `p` to `c` through the relation, so the plan takes `p` from `c`'s entries (a **hop**)
instead of scanning every phone. Each `.requires(...)` is a conjunct, tested as soon as the symbols it reads are bound.

The relation's `unique("owner", "label")` (in `toolkit.py`) says a phone determines its one entry, so a hop from a phone
to its owner gives **at most one** contact. The planner uses that to order the hops:

In [7]:
Owner = (Predicates.OfPredicate.Builder().name("Owner").symbols({"c": Contact, "p": Phone})
         .requires(Predicates.Contains(p.owners, lambda e: e.owner == c)).create())
query.explain(Owner)

['p: scan Phone', 'c: p.owners to owner, at most one, then 1 test']

## Why it works this way

- **One rule, two uses.** Checking and selecting are the same rule read two ways, so they can't disagree.
- **The rule's shape is the hint.** You don't annotate a query for performance. A relation between symbols, said as a
  `Contains`, is a hop; a relation's `unique` clause says how far it fans out. `explain` shows what was made of it.
- **Lazy, over the store's data.** A query reads as you iterate, and sees what the store's roots reach, the same
  data a snapshot would save.